# Lecture 8 — Continuation Value is All You Need

**Computational Methods for Heterogeneous-Agent Macro**
Jeffrey Sun, University of Toronto

---

In L7 we solved Krusell–Smith with an Approximate Law of Motion (ALM): a one-line log-linear rule on $K = \int b\, d\mu$. Here we drop the ALM *and* the scalar-$K$ summary. The network we train, $\widehat{V}^{\mathrm{end}}_\theta$, conditions on the full population distribution $\Lambda$ directly via a learned generalized moment. Training minimises the outer Bellman residual.

## 0 · Setup

Activate the project environment and load packages. **L8 needs `Flux`** for the neural network; we add it if it isn't already installed.

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()

using Flux
using Flux: Chain, Dense, swish, Adam, withgradient, mse
using HouseholdStages
using Printf
using Statistics
using LinearAlgebra: I
using Random: MersenneTwister, seed!
using Plots

  Activating project at `~/projects/research/CV is all you need`


## 1 · The K-S setup we inherit from L7

We copy L7's `KSParams`, prices, and effective labour helpers verbatim. The household block is the *3-stage* simulation chain — `z_shock ∘ income ∘ savings` — which is the within-period solver $\Phi$. L7's 4D chain added `K_evolve ∘ Z_shock` at the tail to encode the agent's ALM belief; today we *drop* that tail. The network encodes the agent's expectations over the next-period $(\Lambda, Z)$ directly.

In [10]:
@kwdef struct KSParams
    β::Float64 = 0.96
    γ::Float64 = 1.0
    α::Float64 = 0.36
    δ::Float64 = 0.025
    z_grid::Vector{Float64} = [0.07, 1.0]
    P_z::Matrix{Float64}    = [0.6   0.4;
                               0.05  0.95]
    Z_vals::Vector{Float64} = [0.99, 1.01]
    P_Z::Matrix{Float64}    = [0.875 0.125;
                               0.125 0.875]
    N_w::Int       = 300
    w_min::Float64 = 0.0
    w_max::Float64 = 80.0
end

const p = KSParams()
N_z = length(p.z_grid)
N_Z = length(p.Z_vals)
@printf "β = %.3f, γ = %.2f, α = %.2f, δ = %.3f, N_w = %d, N_z = %d, N_Z = %d\n" p.β p.γ p.α p.δ p.N_w N_z N_Z

β = 0.960, γ = 1.00, α = 0.36, δ = 0.025, N_w = 300, N_z = 2, N_Z = 2


In [11]:
"Compute stationary mean of idiosyncratic z process"
function ks_effective_labor(P_z::AbstractMatrix, z_grid::AbstractVector)
    n = size(P_z, 1)
    A = P_z' - I(n)
    A[end, :] .= 1.0
    rhs = zeros(n); rhs[end] = 1.0
    π = A \ rhs
    return sum(z_grid .* π)
end

function ks_prices(K::Real, Z::Real, p::KSParams)
    L = ks_effective_labor(p.P_z, p.z_grid)
    r = p.α * Z * (K/L)^(p.α - 1) - p.δ
    w = (1 - p.α) * Z * (K/L)^p.α
    return (; r, w)
end

const L_eff = ks_effective_labor(p.P_z, p.z_grid)
@printf "L_eff = %.4f\n" L_eff

L_eff = 0.8967


## 2 · $\Phi$ in code: the 3-stage household chain

L7's `ks_household_2d` *is* our $\Phi$. Given a $(b, z)$ value function `V_end`, one backward sweep produces `V_start`; one forward sweep pushes `Λ_start` to `Λ_end`. Prices $(r, w)$ are read from `env`. Because the model is single-agent in each period, $\Phi$ depends on $\Lambda$ only through prices — and prices depend on $\Lambda$ only through the scalar $K = \int b\, d\mu$.

In [12]:
_u_crra(c, ::Union{Val{1}, Val{1.0}}) = log(c)
_u_crra(c, ::Val{σ}) where σ = (c^(1 - σ)) / (1 - σ)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

"Build household block"
function ks_household_2d(p::KSParams)
    # Define layout
    layout = StateLayout(
        StateAxis(:wealth, continuous_grid(p.w_min, p.w_max; length=p.N_w, spacing=:log)),
        StateAxis(:z, p.z_grid),
    )

    # Define three stages
    z_shock = MarkovStage(layout; axis=:z, transition=p.P_z)
    income  = WealthChangeStage(layout;
        wealth_post=(cell; env) -> (1 + env.r) * cell.wealth + env.w * cell.z)
    savings = ConsumptionSavingsStage(layout; β=p.β,
        utility=(cell, c; env) -> u_crra(c, Val(p.γ)),
        monotone_search=:divide_conquer)
    
    # Build household chain
    hh = z_shock ∘ income ∘ savings

    # Define moments
    return define_moments!(hh;
                           K_supplied=at_end(integrand=:wealth, reduce=sum))
end

hh = ks_household_2d(p)
@printf "Built 3-stage chain hh = z_shock ∘ income ∘ savings.\n"

Built 3-stage chain hh = z_shock ∘ income ∘ savings.


## 3 · Deterministic Aiyagari SS at $Z = 1$

We need two artefacts from the deterministic problem:

1. **$V_{\mathrm{ss}}(b, z)$** — used as the pretraining target across sampled $\Lambda$'s.
2. **$\Lambda_{\mathrm{ss}}(b, z)$** — the deterministic stationary distribution, used as the initial condition for the ergodic sampler and as the anchor around which we perturb during pretraining.

This is L7's `aiyagari_steady_state_at_Z`, lifted verbatim.

In [16]:
"Solve Aiyagari steady state."
function aiyagari_steady_state_at_Z(p::KSParams; Z::Float64=1.0, verbosity=0, maxiter=1000, rtol=1e-2, update_speed=0.01)

    # Build household
    hh = ks_household_2d(p)

    # Initial guesses
    K = 12.0
    V, Λ = nothing, nothing

    # Loop until convergence
    K_err = Inf
    iters = 0

    while K_err > rtol
        # Compute moments as a function of env
        env = make_env(hh; ks_prices(K, Z, p)...)
        (;V, Λ, moments) = solve_steady_state_given_env!(hh, env; V_init=V, Λ_init=Λ, lambda_tol=1e-5, lambda_maxiter=50_000)
        (;K_supplied) = moments

        # Compute error
        K_err = abs(K_supplied - K) / K
        verbosity >= 1 && @printf "  iter %d: K = %.3f → K_sup = %.3f\n" iters K K_supplied

        # Update K
        K += update_speed * (K_supplied - K)

        # Check maximum iterations
        iters += 1
        iters >= maxiter && error("aiyagari_steady_state_at_Z: did not converge")
    end

    return (; K, Λ, V, iters)
end

println("Solving deterministic Aiyagari SS at Z = 1.0...")
@time det_ss = aiyagari_steady_state_at_Z(p; verbosity=1)
const K_bar = det_ss.K
const V_ss  = det_ss.V
const Λ_ss  = det_ss.Λ
@printf "K̄ = %.4f in %d iters; sum(Λ_ss) = %.6f\n" K_bar det_ss.iters sum(Λ_ss)

Solving deterministic Aiyagari SS at Z = 1.0...
  iter 0: K = 12.000 → K_sup = 44.949
  iter 1: K = 12.329 → K_sup = 39.384
  iter 2: K = 12.600 → K_sup = 23.659
  iter 3: K = 12.711 → K_sup = 20.374
  iter 4: K = 12.787 → K_sup = 18.645
  iter 5: K = 12.846 → K_sup = 17.290
  iter 6: K = 12.890 → K_sup = 16.459
  iter 7: K = 12.926 → K_sup = 15.324
  iter 8: K = 12.950 → K_sup = 14.460
  iter 9: K = 12.965 → K_sup = 14.337
  iter 10: K = 12.979 → K_sup = 14.204
  iter 11: K = 12.991 → K_sup = 14.087
  iter 12: K = 13.002 → K_sup = 13.881
  iter 13: K = 13.011 → K_sup = 13.798
  iter 14: K = 13.019 → K_sup = 13.710
  iter 15: K = 13.026 → K_sup = 13.633
  iter 16: K = 13.032 → K_sup = 13.572
  iter 17: K = 13.037 → K_sup = 12.717
  iter 18: K = 13.034 → K_sup = 12.583
  iter 19: K = 13.029 → K_sup = 12.510
  iter 20: K = 13.024 → K_sup = 12.506
  iter 21: K = 13.019 → K_sup = 12.481
  iter 22: K = 13.014 → K_sup = 12.451
  iter 23: K = 13.008 → K_sup = 12.441
  iter 24: K = 13.002 → K_

## 4 · The reframe: from $K$-ALM to $V_\theta(b, z, \Lambda, Z)$

L7's K-S baseline solved a 4D Bellman by tabulating $V[b, z, K, Z]$ over a small $K$-grid and a 2-point $Z$-grid, assuming the agent forecasts $K_{t+1}$ via a log-linear ALM. The whole *point* of the scalar $K$ was bookkeeping: K-S needed something tractable to forecast, and one number was all the parametric ansatz could carry.

We don't need that compromise. The network can condition on the full distribution $\Lambda$ directly:

$$V_\theta : (b, z, \Lambda, Z) \;\longmapsto\; v \in \mathbb{R}.$$

The function we want is the one that satisfies the Bellman equation. Equivalently: the fixed point of the **outer Bellman operator** $\mathcal{L}$, defined in §7. So we minimise the mean-squared **outer Bellman residual**:

$$\mathcal{J}(\theta) \;=\; \mathbb{E}_{\Lambda^{\mathrm{start}}}\!\bigl[\bigl\|V_\theta - \mathcal{L} V_\theta\bigr\|^2\bigr].$$

The challenge is that $\Lambda$ is high-dimensional: an `N_w × N_z = 80 × 2 = 160`-cell field. We can't feed it directly into a dense net and learn anything meaningful. §5 explains how the **generalized moment** construction handles this — an architecture that summarises $\Lambda$ as a small *learned* vector before the value head sees it.

## 5 · The neural network $V_\theta$: a generalized-moment architecture

$V_\theta(b, z, \Lambda, Z)$ factors into four pieces, mirroring Han–Yang–E (2025):

1. **`pop_agg_net`** $: (b, z) \mapsto \mathbb{R}^{k}$ — embeds each idiosyncratic cell into a $k$-vector. Same network applied to every cell.
2. **Generalized moment** $\mathrm{GM}(\Lambda) \;=\; \sum_{(b, z)} \Lambda(b, z) \cdot \texttt{pop\_agg\_net}(b, z) \;\in\; \mathbb{R}^{k}$ — the population-weighted average of the cell embeddings. One matmul against $\mathrm{vec}(\Lambda)$. **This is how the full distribution gets summarised: as a small $k$-dim learned vector, not a hand-picked scalar.**
3. **`V_pre`** $: (b, z) \mapsto \mathbb{R}^{m}$ — cell-level features that will be combined additively with $\mathrm{GM}$. Same network applied to every cell.
4. **`V_post`** $: \mathbb{R}^{m + N_Z} \mapsto \mathbb{R}$ — combines `V_pre`'s cell features with the projected $\mathrm{GM}$ and the $Z$ one-hot, and reads off $\widehat{V}^{\mathrm{end}}$.

The combination step is **additive**: `V_pre(cell) + gm_post(GM(Λ))`. Then concatenate the $Z$ one-hot and pass through `V_post`. Additive combine forces the network to express $\Lambda$-dependence on the same scale as $(b, z)$-dependence, which stabilises training (Han et al. 2025 §4).

The bottleneck $k$ (set to 16 below) is what makes this scale: we replace an `N_w × N_z`-dim input with a $k$-dim one, *learning* the summary rather than hand-picking $K$.

In [ ]:
# Pre-build the (2, N_w * N_z) idiosyncratic-feature matrix once. Each
# column is the normalized (b, z) coordinates of one (wealth, z) cell.
function build_hh_feat(p::KSParams)
    b_norm = collect(range(0f0, 1f0, length=p.N_w))
    z_norm = collect(range(0f0, 1f0, length=length(p.z_grid)))
    feat = zeros(Float32, 2, p.N_w * length(p.z_grid))
    col = 1
    for j in 1:length(p.z_grid), i in 1:p.N_w
        feat[1, col] = b_norm[i]
        feat[2, col] = z_norm[j]
        col += 1
    end
    return feat
end

const HH_FEAT = build_hh_feat(p)
const N_CELLS = size(HH_FEAT, 2)

# Pre-build one-hot Z rows for each Z_idx. (N_Z, N_CELLS) shape so we can
# vcat directly with the (mid, N_CELLS) hidden features in V_post.
const Z_ROWS = [repeat(Float32.(1:N_Z .== Z_idx), 1, N_CELLS) for Z_idx in 1:N_Z]

2-element Vector{Matrix{Float32}}:
 [1.0 1.0 … 1.0 1.0; 0.0 0.0 … 0.0 0.0]
 [0.0 0.0 … 0.0 0.0; 1.0 1.0 … 1.0 1.0]

In [18]:
# V_θ : (Λ, Z_idx) ↦ V_end matrix of size N_w × N_z. The output bias of
# V_post is initialised to a rough V_ss mean so the inner backward sweep
# doesn't diverge from random init; pretraining (§10) refines from there.
struct VNet{A, P, G, V}
    pop_agg_net::A    # 2 → gm_dim          per-cell embedding for GM
    V_pre::P          # 2 → mid             per-cell features for V head
    gm_post::G        # gm_dim → mid        project GM to mid space
    V_post::V         # mid + N_Z → 1       combine + Z, read off V_end
end

function VNet(; gm_dim::Int=16, mid::Int=32, seed::Int=9483, v_bias::Float32=20f0)
    seed!(seed)
    pop_agg_net = Chain(
        Dense(2 => mid, swish),
        Dense(mid => gm_dim),
    )
    V_pre = Chain(
        Dense(2 => mid, swish),
        Dense(mid => mid, swish),
    )
    gm_post = Dense(gm_dim => mid)
    V_post = Chain(
        Dense(mid + N_Z => mid, swish),
        Dense(mid => 1),
    )
    V_post.layers[end].bias .= v_bias
    return VNet(pop_agg_net, V_pre, gm_post, V_post)
end

Flux.@layer VNet

function (net::VNet)(Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    pop_emb = net.pop_agg_net(HH_FEAT)        # (gm_dim, N_CELLS)
    h_pre   = net.V_pre(HH_FEAT)              # (mid, N_CELLS)
    gm      = pop_emb * Float32.(vec(Λ))      # (gm_dim,)   — pop-weighted mean
    h_gm    = net.gm_post(gm)                 # (mid,)
    h       = h_pre .+ h_gm                   # broadcast → (mid, N_CELLS)
    pred    = net.V_post(vcat(h, Z_ROWS[Z_idx]))   # (1, N_CELLS)
    return reshape(pred, p.N_w, length(p.z_grid))
end

vnet = VNet()
V0 = vnet(Λ_ss, 1, p)
@printf "VNet output at (Λ_ss, Z=1): shape = %s, range = [%.3f, %.3f]\n" string(size(V0)) minimum(V0) maximum(V0)

VNet output at (Λ_ss, Z=1): shape = (300, 2), range = [19.788, 19.912]


## 6 · One inner solve under $V_\theta$

Given an aggregate state $(\Lambda, Z)$:

1. Evaluate $V_\theta(\Lambda, Z)$ to get an `N_w × N_z` array of $V^{\mathrm{end}}$ values.
2. Build `env` at prices $(r, w) = $ `ks_prices(K(Λ), Z, p)`, where $K(\Lambda) = \int b\, d\Lambda$. Prices depend on $\Lambda$ only through $K$ — that's the production technology, not the network's architecture.
3. One backward sweep through the 3-stage chain returns $V^{\mathrm{start}}$.

This is $\Phi_V$ from the slides (§3).

In [19]:
# Integrate wealth (the K moment) against a 2D distribution.
function integrate_K(hh, Λ::AbstractMatrix)
    b_grid = axisvalues(first(hh.spec.stages).input_layout.axes[1])
    return sum(Λ[i, j] * b_grid[i] for i in axes(Λ, 1), j in axes(Λ, 2))
end

# Run one backward sweep of the 3-stage chain under V_θ at the given (Λ, Z).
# Returns the start-of-period V (an N_w × N_z matrix).
function inner_solve_backward(hh, vnet::VNet, Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    V_end = Float64.(vnet(Λ, Z_idx, p))
    env   = make_env(hh; ks_prices(integrate_K(hh, Λ), p.Z_vals[Z_idx], p)...)
    return backward!(hh, V_end, env)
end

V_start_demo = inner_solve_backward(hh, vnet, Λ_ss, 1, p)
@printf "Inner backward at (Λ_ss, Z=bad): V_start shape = %s, range = [%.3f, %.3f]\n" string(size(V_start_demo)) minimum(V_start_demo) maximum(V_start_demo)

Inner backward at (Λ_ss, Z=bad): V_start shape = (300, 2), range = [17.992, 23.521]


## 7 · The lookahead operator $\mathcal{L} V_\theta$

For a single sampled $\Lambda^{\mathrm{start}}_i$ with aggregate index $Z_i$:

1. Evaluate $V_\theta(\Lambda^{\mathrm{start}}_i, Z_i)$ to get $V^{\mathrm{end}}$ — the agent's current belief about end-of-period continuation values.
2. Push $\Lambda^{\mathrm{start}}_i$ forward one period through $\Phi$ to get $\Lambda^{\mathrm{end}}_i$.
3. For each next-period aggregate state $Z_{\mathrm{next}}$:
   - Run $\Phi_V$ at $(\Lambda^{\mathrm{end}}_i, Z_{\mathrm{next}})$ to get $V^{\mathrm{start}}_{\mathrm{next}}$.
4. Take expectation over $Z_{\mathrm{next}}$ using $P_Z[Z_i, \cdot]$.

Note: in this setup $\Omega(\Lambda^{\mathrm{end}}, Z')$ is the identity — the aggregate shock $Z'$ affects next-period prices and hence the *next* period's $\Phi$, but not $\Lambda$ at the period boundary. So $\Lambda^{\mathrm{start}}_{i+1} = \Lambda^{\mathrm{end}}_i$.

The result is the **lookahead label** $\widehat{V}^{\mathrm{end}}_i \equiv (\mathcal{L} V_\theta)(\Lambda^{\mathrm{start}}_i, Z_i)$.

In [20]:
# Compute the lookahead label LV_θ(Λ_start, Z) — see §7 of the slides.
# Note: requires a *pretrained* V_θ; on a random network the savings policy
# can be degenerate and forward! may produce a collapsed Λ_end. We demo
# this function after pretraining in §10.
function lookahead(hh, vnet::VNet, Λ_start::AbstractMatrix, Z_idx::Int, p::KSParams)
    # Use neural network to get V_end at current aggregate
    V_end_now = Float64.(vnet(Λ_start, Z_idx, p))
    
    # Simulate forward one period.
    env_now   = make_env(hh; ks_prices(integrate_K(hh, Λ_start), p.Z_vals[Z_idx], p)...)
    _         = backward!(hh, V_end_now, env_now)         # seats savings policy
    Λ_end     = forward!(hh, Λ_start)

    # Simulate forward again and iterate backward for each Z_next, then take expectation.
    V_end_label = zeros(Float64, p.N_w, length(p.z_grid))
    for Z_next in 1:N_Z
        V_start_next = inner_solve_backward(hh, vnet, Λ_end, Z_next, p)
        V_end_label .+= p.P_Z[Z_idx, Z_next] .* V_start_next
    end
    return (; V_end_label, Λ_end)
end

lookahead (generic function with 1 method)

## 8 · The outer Bellman residual loss

For a batch of samples $\{(\Lambda^{\mathrm{start}}_i, Z_i)\}$, compute the lookahead label $\widehat{V}^{\mathrm{end}}_i$ once *outside* `withgradient`, then fit $V_\theta$ to it in mean-square. This is the standard semi-gradient treatment — we don't differentiate through $\Phi$.

In [21]:
# Build training targets by running the lookahead operator on each sample.
# Returns a vector of (Λ, Z_idx, target) tuples; targets are stop-gradient.
function build_targets(hh, vnet::VNet, batch, p::KSParams)
    targets = NamedTuple{(:Λ, :Z_idx, :target), Tuple{Matrix{Float64}, Int, Matrix{Float32}}}[]
    for (Λ_start, Z_idx) in batch
        out = lookahead(hh, vnet, Λ_start, Z_idx, p)
        push!(targets, (; Λ=copy(Λ_start), Z_idx, target=Float32.(out.V_end_label)))
    end
    return targets
end

# Outer Bellman residual loss on a batch of (Λ, Z, target) tuples.
function bellman_residual_loss(vnet::VNet, targets, p::KSParams)
    L = 0f0
    for (Λ, Z_idx, target) in targets
        pred = vnet(Λ, Z_idx, p)
        # Normalize by the target's own scale so the loss is dimensionless.
        scale = max(var(target), 1f-6)
        L += sum((pred .- target).^2) / (length(pred) * scale)
    end
    return L / length(targets)
end

bellman_residual_loss (generic function with 1 method)

## 9 · Sampling $\Lambda^{\mathrm{start}}$

MCMC-style: simulate the model under the *current* $V_\theta$, discard burn-in, subsample. The sampler tracks the network — early in training, samples are drawn from the trajectory induced by an under-trained network; the distribution improves with $\theta$.

In [22]:
# Draw one period's Z' | Z from the aggregate Markov chain.
function sample_Z(Z_idx::Int, P_Z::AbstractMatrix, rng)
    probs = P_Z[Z_idx, :]
    u = rand(rng)
    s = 0.0
    for j in eachindex(probs)
        s += probs[j]
        u <= s && return j
    end
    return length(probs)
end

# Simulate one period forward under V_θ at realised (Λ_t, Z_t). Reseats the
# savings policy from V_θ first; returns Λ_{t+1} (= Λ_end).
function sim_one_step(hh, vnet::VNet, Λ::AbstractMatrix, Z_idx::Int, p::KSParams)
    V_end = Float64.(vnet(Λ, Z_idx, p))
    env   = make_env(hh; ks_prices(integrate_K(hh, Λ), p.Z_vals[Z_idx], p)...)
    backward!(hh, V_end, env)
    return forward!(hh, Λ)
end

# MCMC-style sample of (Λ_start, Z) along the ergodic path induced by V_θ.
function sample_ergodic(hh, vnet::VNet, p::KSParams;
                        T::Int=200, burn::Int=100, every::Int=2,
                        rng=MersenneTwister(8421))
    Λ      = copy(Λ_ss)
    Z_idx  = 1
    batch  = Tuple{Matrix{Float64}, Int}[]
    for t in 1:T
        Λ     = sim_one_step(hh, vnet, Λ, Z_idx, p)
        Z_idx = sample_Z(Z_idx, p.P_Z, rng)
        if t > burn && (t - burn) % every == 0
            push!(batch, (copy(Λ), Z_idx))
        end
    end
    return batch
end

sample_ergodic (generic function with 1 method)

## 10 · Pretrain: tile the deterministic $V_{\mathrm{ss}}$ across $(\Lambda, Z)$

A randomly-initialised $V_\theta$ often produces within-period solves that fail numerically (consumption tries to go off-grid, savings policy degenerates). The cheap fix: pretrain $V_\theta$ to match the deterministic-SS value function $V_{\mathrm{ss}}$ across a *cloud* of $\Lambda$'s (perturbed $\Lambda_{\mathrm{ss}}$) and across all $Z$'s. The network ends up on a sane manifold; the residual training loop refines from there.

The point is *not* that $V_{\mathrm{ss}}$ is the right answer at any $\Lambda$ — it isn't, except by coincidence at $\Lambda = \Lambda_{\mathrm{ss}}, Z = 1$. The point is that it's a sensible *constant* baseline, which gets the network into the right scale and shape before real training.

In [ ]:
# Cheap Λ-perturbation: multiplicative noise + renormalise. Used to give
# pretraining a small cloud of plausible distributions around Λ_ss.
function perturb_Λ(Λ::AbstractMatrix, σ::Float64, rng)
    Λp = Λ .* (1 .+ σ .* randn(rng, size(Λ)))
    Λp = max.(Λp, 0)
    return Λp ./ sum(Λp)
end

# Pretrain V_θ on the deterministic SS V_ss, replicated across a cloud of
# (Λ, Z) inputs. Returns the loss history.
function pretrain_to_ss!(vnet::VNet, V_ss_target::AbstractMatrix, p::KSParams;
                         epochs::Int=300, lr::Float64=5e-3,
                         n_Λ::Int=6, σ::Float64=0.10, seed::Int=4747)
    opt    = Flux.setup(Adam(lr), vnet)
    target = Float32.(V_ss_target)
    rng    = MersenneTwister(seed)
    hist   = Float64[]
    for epoch in 1:epochs
        Λ_cloud = [perturb_Λ(Λ_ss, σ, rng) for _ in 1:n_Λ]
        loss, grads = withgradient(vnet) do net
            L = 0f0
            for Λ in Λ_cloud, Z_idx in 1:N_Z
                pred = net(Λ, Z_idx, p)
                L += sum((pred .- target).^2)
            end
            L / (n_Λ * N_Z * length(target))
        end
        Flux.update!(opt, vnet, grads[1])
        push!(hist, Float64(loss))
    end
    return hist
end

println("Pretraining V_θ on V_ss across a Λ-cloud...")
@time pre_hist = pretrain_to_ss!(vnet, V_ss, p; epochs=300)
@printf "Pretrain loss: %.4e → %.4e (over %d epochs)\n" pre_hist[1] pre_hist[end] length(pre_hist)

## 11 · The training loop

One epoch = sample a fresh batch under the current $V_\theta$, compute the lookahead labels (stop-gradient), take one Adam step on the squared residual. This is the simplest semi-gradient form; the optimised implementation in `reference_materials/example_usage/stochastic_transition/` uses cosine-annealed learning rates, larger batches, multiple gradient steps per sample, and per-location backprop.

In [ ]:
# Outer-Bellman-residual training loop. Returns the loss history.
function train!(vnet::VNet, hh, p::KSParams;
                epochs::Int=50, lr::Float64=1e-3,
                T::Int=200, burn::Int=100, every::Int=4)
    opt  = Flux.setup(Adam(lr), vnet)
    hist = Float64[]
    rng  = MersenneTwister(1729)
    for epoch in 1:epochs
        batch   = sample_ergodic(hh, vnet, p; T, burn, every, rng=MersenneTwister(rand(rng, UInt32)))
        targets = build_targets(hh, vnet, batch, p)
        loss, grads = withgradient(vnet) do net
            bellman_residual_loss(net, targets, p)
        end
        Flux.update!(opt, vnet, grads[1])
        push!(hist, Float64(loss))
        if epoch == 1 || epoch % 5 == 0
            @printf "  epoch %3d / %3d: loss = %.4e (batch = %d)\n" epoch epochs loss length(batch)
        end
    end
    return hist
end

println("Running 50-epoch CVIAYN training (CPU demonstration, NOT a converged solve)...")
@time train_hist = train!(vnet, hh, p; epochs=50)

## 12 · Reading the answer

Two plots: the loss curve, and $V_\theta$ vs. the deterministic-SS $V$ at a few sample $(\Lambda, Z)$.

In [ ]:
# Loss curve (semilog). A converged solve would push this many decades lower.
plot(train_hist;
     yaxis=:log, lw=2, color=:steelblue,
     xlabel="epoch", ylabel="outer Bellman residual",
     title="CVIAYN training curve (50 epochs, CPU demo)",
     label="loss", size=(720, 360), legend=:topright)

In [ ]:
# Compare V_θ(b, z=high) at three (Λ, Z) settings with the deterministic SS.
# We perturb Λ_ss to show that V_θ adapts to the distributional input.
rng_show = MersenneTwister(12345)
Λ_high   = perturb_Λ(Λ_ss, 0.20, rng_show)
Λ_low    = perturb_Λ(Λ_ss, 0.20, rng_show)
b_grid = axisvalues(first(hh.spec.stages).input_layout.axes[1])
plt = plot(; xlabel="wealth b", ylabel="V_θ(b, z=high)",
           title="V_θ vs. deterministic SS V (z = high)",
           size=(720, 360))
plot!(plt, b_grid, V_ss[:, end]; lw=2, color=:black, ls=:dash, label="V_ss (det.)")
for (Λ_show, Z_show, lbl) in [(Λ_ss, 1, "Λ_ss, Z=bad"), (Λ_ss, 2, "Λ_ss, Z=good"), (Λ_high, 1, "perturbed Λ, Z=bad")]
    V_show = vnet(Λ_show, Z_show, p)
    plot!(plt, b_grid, V_show[:, end]; lw=2, label=lbl)
end
plt

## 13 · Comparison with the L7 K-S baseline

The deep comparison — running the converged K-S baseline from L7, computing $V$-MSE on a common $(b, z, \Lambda, Z)$ proxy grid, evaluating Den Haan errors under each — belongs in `paper_code/` (the horse-race driver). It is *deferred* in this notebook per session scope.

What we can do here cheaply: summarise where $V_\theta$ ended up so the loss-curve plot has context.

In [ ]:
# Pointer cell — actual comparison driver lives in paper_code/.
# We summarise where V_θ ended up so the loss-curve cell has context.
final_loss = train_hist[end]
@printf "Final outer Bellman residual after %d epochs: %.4e\n" length(train_hist) final_loss
@printf "(For reference: the paper's Peking 2025 §5 figure reports K-S baseline plateau\n"
@printf " near 10^-3.2 and CVIAYN converged near 10^-4.4 in relative-variance units —\n"
@printf " each with several thousand epochs on a GPU.)\n"